# SupportPilot - LoRA fine-tune (Kaggle)
Settings: Accelerator = GPU (T4 / P100), Internet = On. Attach your `supportpilot-data` dataset.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # use ONE gpu (avoids multi-GPU DataParallel surprises on T4 x2)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
print(torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
!pip install -q -U transformers datasets peft trl accelerate
!pip list 2>/dev/null | grep -E "^(transformers|datasets|peft|trl|accelerate) "

In [ ]:
import glob
train_path = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]
eval_path  = glob.glob("/kaggle/input/**/eval.jsonl",  recursive=True)[0]
print(train_path, eval_path)

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"     # if you hit out-of-memory: "Qwen/Qwen2.5-0.5B-Instruct"

dataset = load_dataset("json", data_files={"train": train_path, "eval": eval_path})
print(dataset)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
model.config.use_cache = False

In [ ]:
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
def format_example(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

train_ds = dataset["train"].map(format_example)
eval_ds  = dataset["eval"].map(format_example)
print(train_ds[0]["text"][:600])

In [ ]:
training_args = SFTConfig(
    output_dir="/kaggle/working/supportpilot-checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    fp16=True,
    gradient_checkpointing=True,
    max_length=1024,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    processing_class=tokenizer, peft_config=lora_config,
)
trainer.train()

In [ ]:
OUTPUT_DIR = "/kaggle/working/supportpilot-lora"
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
!cd /kaggle/working && zip -r supportpilot-lora.zip supportpilot-lora && ls -lh supportpilot-lora.zip

In [ ]:
# Quick sanity check on one held-out example
trainer.model.config.use_cache = True
msgs = eval_ds[0]["messages"][:2]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
out = trainer.model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))